# Cleaning rules for classification (derived from 200 real samples)

Manual read of 200 random rows from `data/unlabeled_10k.jsonl` (see the
conversation this notebook comes out of) surfaced one dominant, systematic
issue: the `[MENTION]`/`[URL]` anonymization placeholders (inserted by
`clean_text.clean()` upstream in Youtube_scrap/Mountada_djelfa_scrap) are
themselves ASCII/Latin-script tokens. Sitting inside otherwise 100%
Arabic-script text, they're enough to trip a naive script-mix check into
calling the row "mixed" -- and by extension, risk Qwen calling it
`code_switch` for the wrong reason (an anonymization artifact, not a real
second language).

A second, related artifact showed up specifically after `[MENTION]`: a
short reply-anchor ID fragment glued directly onto the next real word with
no space, e.g. `"[MENTION] -xb5ttربي يسعدك"` -- the real word `ربي` starts
immediately after `xb5tt` with no separator.

This notebook: quantifies how much of the `youtube_mixed` sampling bucket
is actually just this artifact (not genuine code-switching), derives a
cleaning regex, verifies it against real before/after examples, and
re-measures script-bucket composition after cleaning. Genuine
code-switched text (real French/English words dropped into Arabic
sentences -- confirmed common in the same 200-sample read) must NOT be
touched by this cleaning; only the placeholder artifacts are in scope.

In [1]:
import json
import re
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "unlabeled_10k.jsonl"
rows = [json.loads(l) for l in DATA_PATH.open("r", encoding="utf-8")]
print(f"{len(rows):,} rows loaded")

ARABIC_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]")
LATIN_RE = re.compile(r"[a-zA-Z\u00C0-\u024F]")


def script_of(text: str) -> str:
    has_ar = bool(ARABIC_RE.search(text))
    has_lat = bool(LATIN_RE.search(text))
    if has_ar and has_lat:
        return "mixed"
    if has_ar:
        return "arabic"
    if has_lat:
        return "latin"
    return "other"

10,000 rows loaded


## 1. Quantify the contamination

Within the `youtube_mixed` sample_group (rows that were bucketed as
"mixed" by the *original*, uncleaned script check when the dataset was
built): how many contain `[MENTION]`/`[URL]`, and of those, how many
would stop being "mixed" if the placeholder were the only Latin content?

In [2]:
PLACEHOLDER_RE = re.compile(r"\[MENTION\]|\[URL\]")

mixed_rows = [r for r in rows if r["sample_group"] == "youtube_mixed"]
has_placeholder = [r for r in mixed_rows if PLACEHOLDER_RE.search(r["text"])]
print(f"youtube_mixed rows: {len(mixed_rows):,}")
print(f"  containing [MENTION]/[URL]: {len(has_placeholder):,} ({len(has_placeholder)/len(mixed_rows):.1%})")

# Of those, how many have NO other Latin-script content once the raw
# placeholder text itself is excluded from the check -- i.e. the
# placeholder is the ONLY reason this row scored "mixed".
naive_stripped_would_flip = 0
for r in has_placeholder:
    without_placeholder = PLACEHOLDER_RE.sub("", r["text"])
    if script_of(without_placeholder) != "mixed":
        naive_stripped_would_flip += 1
print(f"  would flip to non-mixed if [MENTION]/[URL] text itself were excluded: {naive_stripped_would_flip:,}")

youtube_mixed rows: 2,833
  containing [MENTION]/[URL]: 2,127 (75.1%)
  would flip to non-mixed if [MENTION]/[URL] text itself were excluded: 1,538


## 2. The ID-fragment problem

Simply removing the literal `[MENTION]`/`[URL]` strings isn't enough --
a short reply-anchor fragment often rides along right after `[MENTION]`,
glued to the next real word with no space. Show real examples first.

In [3]:
FRAGMENT_AFTER_RE = re.compile(r"\[MENTION\]\s*-[^\s]{1,10}")

examples_with_fragment = [r for r in has_placeholder if FRAGMENT_AFTER_RE.search(r["text"])]
print(f"{len(examples_with_fragment):,} of {len(has_placeholder):,} placeholder rows have the trailing ID-fragment pattern\n")
for r in examples_with_fragment[:12]:
    print(repr(r["text"][:90]))

690 of 2,127 placeholder rows have the trailing ID-fragment pattern

'[MENTION] -x1jهذي كتبتيها من شات جي بي تي نعرف أسلوبو مليح مليح 🥲'
'[MENTION] -sf4mv يعطيك الصحة نتا فهمتني ولخرين تقول دواب'
'[MENTION] -ml6du وانت عايش فالمريخ'
'[MENTION] -c7kانا في ولايتي الحاويات في كل مكان و الشعب ياكل ويرمي وين جات شانقولوا عليهم '
'[MENTION] -zr9yn يا رب'
'[MENTION] -s8p4mفلسفة لافاءدة منها نحن علميين ولسنا أدبيين'
'[MENTION] -uc1ol وأصبح الناس يمثيلون مثل الأفلام أو أكثر لا يوجد رومنسية ولاهم'
'[MENTION] -j1i لدينا أزمة رجال حقيقيين هناك الذكور فقط، لو كان ارباب البيوت قوامون ويتحملو'
'[MENTION] -bacما يفهموش حاسبين رواحهم علماء'
'[MENTION] -Forumمروكي عشرة في عقل %48😂'
'[MENTION] -ت9ز نن 😂'
'[MENTION] -f9rشكون قالك؟ \nممكن الجن مسلم'


## 3. The cleaning function

Strips `[MENTION]`/`[URL]` and, if present, the glued-on ID fragment
immediately after `[MENTION]` (bounded to 1-10 characters -- these
fragments are short by construction; bounding the length keeps this from
ever eating into a real word if a hyphen legitimately starts one). Then
collapses the whitespace left behind. Real embedded French/English words
elsewhere in the text are completely untouched -- this only targets the
placeholder + its attached fragment, nothing else.

In [4]:
MENTION_WITH_FRAGMENT_RE = re.compile(r"\[MENTION\](\s*-[^\s]{1,10})?")
URL_RE = re.compile(r"\[URL\]")
WHITESPACE_RE = re.compile(r"[ \t]+")


def clean_for_classification(text: str) -> str:
    text = MENTION_WITH_FRAGMENT_RE.sub("", text)
    text = URL_RE.sub("", text)
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text

## 4. Before/after on real examples

Manually inspect -- confirms the fragment is removed cleanly and no real
word gets clipped, and that genuine code-switch examples (no placeholder)
pass through completely unchanged.

In [5]:
print("=== placeholder-only rows (should lose all Latin content) ===")
for r in examples_with_fragment[:8]:
    before = r["text"]
    after = clean_for_classification(before)
    print(f"BEFORE: {before[:90]!r}")
    print(f"AFTER:  {after[:90]!r}")
    print(f"  script: {script_of(before)} -> {script_of(after)}")
    print()

print("\n=== genuine code-switch rows (must be unchanged) ===")
genuine_examples = [
    "مين كانت كاينة وهران بزاف مدن و بلدان ودول كانوا عاد تراب فاتوا 100حضارة على بلادنا وما نقص والوا au contraire الجزاءر قعدت هي هي زينة البلدان",
    "أمين بومدين الله ابارك عليك خدمت على روحك ثقافتك و مربي و généreu في المعلومة",
    "Tellement vrai الهدرة هادي قاع نعرفوها و قاع علابالنا بخطورة الوضع",
]
for text in genuine_examples:
    cleaned = clean_for_classification(text)
    assert cleaned == text, "genuine code-switch text must not be modified!"
    print(f"unchanged, script stays: {script_of(text)}")

=== placeholder-only rows (should lose all Latin content) ===
BEFORE: '[MENTION] -x1jهذي كتبتيها من شات جي بي تي نعرف أسلوبو مليح مليح 🥲'
AFTER:  'كتبتيها من شات جي بي تي نعرف أسلوبو مليح مليح 🥲'
  script: mixed -> arabic

BEFORE: '[MENTION] -sf4mv يعطيك الصحة نتا فهمتني ولخرين تقول دواب'
AFTER:  'يعطيك الصحة نتا فهمتني ولخرين تقول دواب'
  script: mixed -> arabic

BEFORE: '[MENTION] -ml6du وانت عايش فالمريخ'
AFTER:  'وانت عايش فالمريخ'
  script: mixed -> arabic

BEFORE: '[MENTION] -c7kانا في ولايتي الحاويات في كل مكان و الشعب ياكل ويرمي وين جات شانقولوا عليهم '
AFTER:  'في ولايتي الحاويات في كل مكان و الشعب ياكل ويرمي وين جات شانقولوا عليهم هادوا؟'
  script: mixed -> arabic

BEFORE: '[MENTION] -zr9yn يا رب'
AFTER:  'يا رب'
  script: mixed -> arabic

BEFORE: '[MENTION] -s8p4mفلسفة لافاءدة منها نحن علميين ولسنا أدبيين'
AFTER:  'لافاءدة منها نحن علميين ولسنا أدبيين'
  script: mixed -> arabic

BEFORE: '[MENTION] -uc1ol وأصبح الناس يمثيلون مثل الأفلام أو أكثر لا يوجد رومنسية ولاهم'
AFTER:  

## 5. Re-measure: how much of the sampling was actually contaminated?

Re-run `script_of()` on the cleaned text for every row and compare to the
original `sample_group` bucket assignment made at dataset-build time (on
*uncleaned* text). This tells us how many `youtube_mixed`-bucketed rows
were never genuinely mixed at all.

In [6]:
from collections import Counter

reclassified = Counter()
for r in mixed_rows:
    cleaned_script = script_of(clean_for_classification(r["text"]))
    reclassified[cleaned_script] += 1

print("youtube_mixed bucket (n={:,}), script after cleaning:".format(len(mixed_rows)))
for script, count in reclassified.most_common():
    print(f"  {script:<8} {count:5,}  ({count/len(mixed_rows):.1%})")

youtube_mixed bucket (n=2,833), script after cleaning:
  arabic   2,028  (71.6%)
  mixed      774  (27.3%)
  other       28  (1.0%)
  latin        3  (0.1%)


## Conclusion / recommendation

If a large share of `youtube_mixed` turns out to be artifact-only (see
the cell above), the practical implication is: **the `youtube_mixed`
sampling bucket in `data/unlabeled_10k.jsonl` was built by bucketing on
*uncleaned* text, so a meaningful fraction of its 2,833 rows are not
genuinely code-switched** -- undermining the reason that bucket exists
(guaranteeing real code_switch representation for labeling). The fix
is to apply `clean_for_classification()` *before* the script-bucketing
step in `build_dataset.py`, not just before sending text to Qwen for
labeling, and rebuild `unlabeled_10k.jsonl` once with that ordering.
That rebuild is NOT done by this notebook -- it only derives and
validates the function; applying it to rebuild the dataset is a
follow-up step to confirm before doing.